# Exercícios Aula 05 – Apache Spark
Food Establishment Inspection Data (King County) – versão atualizada, processada no Amazon EMR com PySpark e Spark SQL.

In [ ]:
from pyspark.sql import functions as F

# Rotas de entrada e saída no Amazon S3
data_source = "s3://bucket-andre-lab1/Food_Establishment_Inspection_Data_20261022.csv"
output_uri = "s3://bucket-andre-lab1/output/"

# Ler tudo como string: inferSchema transformaria Zip Code e Parcel Number em inteiros (perdendo zeros à esquerda)
# e não reconhece a data no formato MM/dd/yyyy
raw_df = spark.read.option("header", "true").csv(data_source)

# Padronizar nomes das colunas para snake_case ("Inspection Date" -> inspection_date)
raw_df = raw_df.toDF(*[c.strip().lower().replace(" ", "_") for c in raw_df.columns])
raw_df.printSchema()

## 1. Estrutura e qualidade dos dados

In [ ]:
# Converter tipos e padronizar texto
restaurants_df = (raw_df
    .withColumn("inspection_date", F.to_date("inspection_date", "MM/dd/yyyy"))
    .withColumn("inspection_score", F.col("inspection_score").cast("int"))
    .withColumn("violation_points", F.col("violation_points").cast("int"))
    .withColumn("risk_category", F.col("risk_category").cast("int"))
    .withColumn("inspection_closed_business", F.col("inspection_closed_business") == "Yes")
    .withColumn("city", F.upper(F.trim("city")))
    .withColumn("violation_type", F.upper(F.trim("violation_type")))
    .cache())

print(f"Registros: {restaurants_df.count()}")
print(f"Colunas: {len(restaurants_df.columns)}")
restaurants_df.printSchema()

In [ ]:
# Valores ausentes por coluna
restaurants_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in restaurants_df.columns]).show(vertical=True)

In [ ]:
# Domínio das colunas categóricas usadas nas análises
restaurants_df.groupBy("violation_type").count().show()
restaurants_df.groupBy("inspection_result").count().show()
restaurants_df.select(F.min("inspection_date"), F.max("inspection_date")).show()

**Alterações em relação ao dataset do laboratório**

| Laboratório | Dataset atualizado | Ajuste no código |
|---|---|---|
| 5 colunas, nomes em snake_case (`name`, `violation_type`, ...) | 20 colunas, nomes com espaços e maiúsculas (`Violation Type`, `Inspection Date`, ...) | Renomear todas as colunas para snake_case com `toDF` |
| Sem data, cidade ou identificador de inspeção | `Inspection Date`, `City`, `Business_ID`, `Inspection_Serial_Num` | Permitem as análises por cidade, por ano e por inspeção única |
| `inferSchema` suficiente | Data em `MM/dd/yyyy` e códigos numéricos com zeros à esquerda (`Zip Code`, `Parcel Number`) | Ler como string e converter explicitamente (`to_date`, `cast("int")`) |
| `inspection_closed_business` = `TRUE/FALSE` | `Yes/No` | Converter para booleano |
| Cidade inexistente | Mesma cidade com grafias diferentes (`Auburn`/`AUBURN`) | `upper(trim(city))` |

Cada linha continua sendo **uma infração** (ou uma inspeção sem infração, com `violation_type` nulo). Por isso uma mesma inspeção aparece em várias linhas.

In [ ]:
restaurants_df.createOrReplaceTempView("restaurant_violations")

## 2. Infrações RED por estabelecimento

In [ ]:
# Agrupar por Business_ID: o nome sozinho junta unidades diferentes de uma mesma rede (ex.: várias lojas SUBWAY)
red_by_establishment = spark.sql("""
    SELECT business_id, first(name) AS name, first(address) AS address, first(city) AS city,
           COUNT(*) AS total_red_violations,
           COUNT(DISTINCT inspection_serial_num) AS inspections_with_red
    FROM restaurant_violations
    WHERE violation_type = 'RED'
    GROUP BY business_id
    ORDER BY total_red_violations DESC
    LIMIT 10
""")
red_by_establishment.show(truncate=False)
red_by_establishment.write.option("header", "true").mode("overwrite").csv(output_uri + "red_by_establishment/")

In [ ]:
# Mesma consulta do laboratório (agrupando por nome), para comparação
spark.sql("""
    SELECT name, COUNT(*) AS total_red_violations
    FROM restaurant_violations
    WHERE violation_type = 'RED'
    GROUP BY name
    ORDER BY total_red_violations DESC
    LIMIT 10
""").show(truncate=False)

## 3. Infrações RED por cidade

In [ ]:
red_by_city = spark.sql("""
    SELECT city, COUNT(*) AS total_red_violations,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_total
    FROM restaurant_violations
    WHERE violation_type = 'RED' AND city IS NOT NULL
    GROUP BY city
    ORDER BY total_red_violations DESC
    LIMIT 10
""")
red_by_city.show(truncate=False)
red_by_city.write.option("header", "true").mode("overwrite").csv(output_uri + "red_by_city/")

In [ ]:
import matplotlib.pyplot as plt

pdf = red_by_city.toPandas()
plt.clf()
plt.figure(figsize=(10, 5))
plt.barh(pdf["city"][::-1], pdf["total_red_violations"][::-1])
plt.xlabel("Registros de infrações RED")
plt.title("Top 10 cidades por infrações RED")
plt.tight_layout()
%matplot plt

## 4. Evolução temporal
Registros de infração (linhas) são diferentes de inspeções únicas (`inspection_serial_num` distintos).

In [ ]:
by_year = spark.sql("""
    SELECT year(inspection_date) AS year,
           COUNT(DISTINCT inspection_serial_num) AS unique_inspections,
           COUNT(violation_type) AS violation_records,
           SUM(CASE WHEN violation_type = 'RED' THEN 1 ELSE 0 END) AS red_records,
           COUNT(DISTINCT CASE WHEN violation_type = 'RED' THEN inspection_serial_num END) AS inspections_with_red
    FROM restaurant_violations
    GROUP BY year(inspection_date)
    ORDER BY year
""")
by_year.show()
by_year.write.option("header", "true").mode("overwrite").csv(output_uri + "by_year/")

In [ ]:
pdf = by_year.toPandas()
plt.clf()
plt.figure(figsize=(10, 5))
for col, label in [("unique_inspections", "Inspeções únicas"), ("red_records", "Registros RED"),
                   ("inspections_with_red", "Inspeções com ao menos 1 RED")]:
    plt.plot(pdf["year"], pdf[col], marker="o", label=label)
plt.xticks(pdf["year"])
plt.xlabel("Ano")
plt.ylabel("Quantidade")
plt.title("Inspeções e infrações RED por ano (2026 parcial, até out/2026)")
plt.legend()
plt.tight_layout()
%matplot plt

## 5. Execução distribuída
Plano físico da consulta da questão 2: os operadores `Exchange` marcam os pontos de shuffle (redistribuição por `business_id` no `GROUP BY` e por ordenação no `ORDER BY ... LIMIT`). Na Spark UI (aba SQL/DataFrame), localize essa consulta e observe os Jobs, Stages, Tasks e Executors correspondentes.

In [ ]:
red_by_establishment.explain(mode="formatted")

In [ ]:
# Informações do cluster para relacionar com a aba Executors
sc = spark.sparkContext
print("Spark UI:", sc.uiWebUrl)
print("Default parallelism:", sc.defaultParallelism)
print("Partições do DataFrame:", restaurants_df.rdd.getNumPartitions())
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))